# 04 - Model Training & Experiment Tracking

This notebook demonstrates baseline model fitting (Logistic Regression), gradient boosted tree training (XGBoost), hyperparameter tuning, and experiment tracking using MLflow without data leakage.

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import mlflow
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

sys.path.append('..')
from src.config import load_config
from src.data_loader import load_raw_data, get_stratified_split
from src.features import build_preprocessing_pipeline
from src.metrics import calculate_credit_metrics
from src.train import train_credit_scoring_system

# Set MLflow tracking URI to project root SQLite DB
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("Credit_Risk_Scoring")
print("MLflow configured for experiment tracking.")

## 1. Load Data & Stratified 70/15/15 Split

Data is split into Train (70%), Validation (15%), and Test (15%) partitions. The holdout test partition is strictly isolated.

In [ ]:
df = load_raw_data()
X_train, y_train, X_val, y_val, X_test, y_test = get_stratified_split(df)

preprocessor = build_preprocessing_pipeline()
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

print(f"X_train shape: {X_train_proc.shape}, X_val shape: {X_val_proc.shape}")

## 2. Baseline Model vs Tuned XGBoost

Evaluate baseline logistic regression vs XGBoost on the validation set.

In [ ]:
# Baseline: Balanced Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_proc, y_train)
y_val_prob_lr = lr.predict_proba(X_val_proc)[:, 1]
print("Baseline Logistic Regression ROC-AUC:", roc_auc_score(y_val, y_val_prob_lr))

# XGBoost Classifier (evaluated on validation set for early stopping, NOT test set)
xgb_clf = xgb.XGBClassifier(max_depth=4, learning_rate=0.05, n_estimators=200, subsample=0.8, random_state=42, eval_metric='auc')
xgb_clf.fit(X_train_proc, y_train, eval_set=[(X_val_proc, y_val)], verbose=False)
y_val_prob_xgb = xgb_clf.predict_proba(X_val_proc)[:, 1]
print("XGBoost Classifier ROC-AUC:", roc_auc_score(y_val, y_val_prob_xgb))

## 3. Run Production Pipeline via CLI Module

Executes the full calibrated training pipeline with MLflow tracking.

In [ ]:
result = train_credit_scoring_system()
print("Training Run ID:", result['run_id'])
print("Optimal Threshold:", result['optimal_threshold'])
print("Validation Metrics:", result['val_metrics'])